# Ames Housing Price Prediction

## Load Packages

In [1]:
import math
import pandas as pd
import numpy as np
from operator import itemgetter


import matplotlib.pyplot as plt
import seaborn as sns


from sklearn.model_selection import train_test_split
import sklearn.metrics as metrics


from sklearn import tree
from sklearn.tree import _tree

from sklearn.ensemble import RandomForestRegressor 
from sklearn.ensemble import RandomForestClassifier 

from sklearn.ensemble import GradientBoostingRegressor 
from sklearn.ensemble import GradientBoostingClassifier

from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs

import warnings
warnings.filterwarnings("ignore")

## Load Data

In [2]:
TARGET_A = "SalePrice"

df = pd.read_csv("data/train.csv")
df_test  = pd.read_csv("data/test.csv")

df.head(5)

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Data Cleaning

In [3]:
# One Hot Encoding and Impute Categorical Value

dt = df.dtypes

objList = []
intList = []
floatList = []
numList = []

for i in dt.index :
   if i in ( [ TARGET_A ] ) : continue
   if dt[i] in (["str"]) : objList.append( i )
   if dt[i] in (["float64"]) : floatList.append( i )
   if dt[i] in (["int64"]) : intList.append( i )
   if dt[i] in (["float64","int64"]) : numList.append( i )

for i in numList :
    if df[i].isna().sum() == 0 : continue
    FLAG = "M_" + i
    IMP = "IMP_" + i
    df[ FLAG ] = df[i].isna() + 0
    df[ IMP ] = df[ i ]
    df.loc[ df[IMP].isna(), IMP ] = df[i].median()
    df = df.drop( i, axis=1 )

for i in objList :
    if df[i].isna().sum() == 0 : continue
    NAME = "IMP_"+i
    df[NAME] = df[i]
    df[NAME] = df[NAME].fillna("MISSING")
    df = df.drop( i, axis=1 )

dt = df.dtypes

objList = []
for i in dt.index :
    if i in ( [ TARGET_A ] ) : continue
    if dt[i] in (["str"]) : objList.append( i )

for i in objList :
    thePrefix = "z_" + i
    y = pd.get_dummies( df[i], dtype = int, prefix=thePrefix, dummy_na=False)   
    df = pd.concat( [df, y], axis=1 )
    df = df.drop( i, axis=1 )

df.head(5)

,Id,MSSubClass,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,BsmtFinSF1,BsmtFinSF2,BsmtUnfSF,...,z_IMP_Fence_GdPrv,z_IMP_Fence_GdWo,z_IMP_Fence_MISSING,z_IMP_Fence_MnPrv,z_IMP_Fence_MnWw,z_IMP_MiscFeature_Gar2,z_IMP_MiscFeature_MISSING,z_IMP_MiscFeature_Othr,z_IMP_MiscFeature_Shed,z_IMP_MiscFeature_TenC
0,1,60,8450,7,5,2003,2003,706,0,150,...,0,0,1,0,0,0,1,0,0,0
1,2,20,9600,6,8,1976,1976,978,0,284,...,0,0,1,0,0,0,1,0,0,0
2,3,60,11250,7,5,2001,2002,486,0,434,...,0,0,1,0,0,0,1,0,0,0
3,4,70,9550,7,5,1915,1970,216,0,540,...,0,0,1,0,0,0,1,0,0,0
4,5,60,14260,8,5,2000,2000,655,0,490,...,0,0,1,0,0,0,1,0,0,0


In [4]:
# Remove Outliers

dt = df.dtypes
numList = []
for i in dt.index :
    if i in ( [ TARGET_A ] ) : continue
    if dt[i] in (["float64","int64"]) : numList.append( i )


for i in numList :
    theMean = df[i].mean()
    theSD = df[i].std()
    theMax = df[i].max()
    theCutoff = round( theMean + 3*theSD )
    if theMax < theCutoff : continue
    FLAG = "O_" + i
    TRUNC = "TRUNC_" + i
    df[ FLAG ] = ( df[i] > theCutoff )+ 0
    df[ TRUNC ] = df[ i ]
    df.loc[ df[TRUNC] > theCutoff, TRUNC ] = theCutoff
    df = df.drop( i, axis=1 )

df.head(5)

,Id,YearBuilt,YearRemodAdd,MoSold,YrSold,SalePrice,IMP_GarageYrBlt,z_MSZoning_RL,z_LotShape_IR1,z_LotShape_Reg,...,O_z_IMP_Fence_MnWw,TRUNC_z_IMP_Fence_MnWw,O_z_IMP_MiscFeature_Gar2,TRUNC_z_IMP_MiscFeature_Gar2,O_z_IMP_MiscFeature_Othr,TRUNC_z_IMP_MiscFeature_Othr,O_z_IMP_MiscFeature_Shed,TRUNC_z_IMP_MiscFeature_Shed,O_z_IMP_MiscFeature_TenC,TRUNC_z_IMP_MiscFeature_TenC
0,1,2003,2003,2,2008,208500,2003.0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,2,1976,1976,5,2007,181500,1976.0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,3,2001,2002,9,2008,223500,2001.0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
3,4,1915,1970,2,2006,140000,1998.0,1,1,0,...,0,0,0,0,0,0,0,0,0,0
4,5,2000,2000,12,2008,250000,2000.0,1,1,0,...,0,0,0,0,0,0,0,0,0,0


In [5]:
# Drop Id

df = df.drop("Id", axis=1)

df.head(5)

,YearBuilt,YearRemodAdd,MoSold,YrSold,SalePrice,IMP_GarageYrBlt,z_MSZoning_RL,z_LotShape_IR1,z_LotShape_Reg,z_LandContour_Lvl,...,O_z_IMP_Fence_MnWw,TRUNC_z_IMP_Fence_MnWw,O_z_IMP_MiscFeature_Gar2,TRUNC_z_IMP_MiscFeature_Gar2,O_z_IMP_MiscFeature_Othr,TRUNC_z_IMP_MiscFeature_Othr,O_z_IMP_MiscFeature_Shed,TRUNC_z_IMP_MiscFeature_Shed,O_z_IMP_MiscFeature_TenC,TRUNC_z_IMP_MiscFeature_TenC
0,2003,2003,2,2008,208500,2003.0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
1,1976,1976,5,2007,181500,1976.0,1,0,1,1,...,0,0,0,0,0,0,0,0,0,0
2,2001,2002,9,2008,223500,2001.0,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
3,1915,1970,2,2006,140000,1998.0,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0
4,2000,2000,12,2008,250000,2000.0,1,1,0,1,...,0,0,0,0,0,0,0,0,0,0


In [6]:
# remove missing values 
df_clean = df[df['SalePrice'].notna()].copy()

# Log transform the target
df_clean['SalePrice_log'] = np.log(df_clean['SalePrice'])

# rename log price as target
TARGET_A = 'SalePrice_log'

X = df_clean.drop(['SalePrice', 'SalePrice_log'], axis=1)
Y = df_clean[['SalePrice_log']]

# Split and train normally
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, train_size=0.8, test_size=0.2, random_state=5
)

# Renmae variables
W_train = X_train
Z_train = Y_train
W_test = X_test
Z_test = Y_test

# After predictions, transform back:
# predictions_actual = np.exp(predictions_log)


In [ ]:
# MODEL ACCURACY METRICS

def getProbAccuracyScores( NAME, MODEL, X, Y ) :
    pred = MODEL.predict( X )
    probs = MODEL.predict_proba( X )
    acc_score = metrics.accuracy_score(Y, pred)
    p1 = probs[:,1]
    fpr, tpr, threshold = metrics.roc_curve( Y, p1)
    auc = metrics.auc(fpr,tpr)
    return [NAME, acc_score, fpr, tpr, auc]

def print_ROC_Curve( TITLE, LIST ) :
    fig = plt.figure(figsize=(6,4))
    plt.title( TITLE )
    for theResults in LIST :
        NAME = theResults[0]
        fpr = theResults[2]
        tpr = theResults[3]
        auc = theResults[4]
        theLabel = "AUC " + NAME + ' %0.2f' % auc
        plt.plot(fpr, tpr, label = theLabel )
    plt.legend(loc = 'lower right')
    plt.plot([0, 1], [0, 1],'r--')
    plt.xlim([0, 1])
    plt.ylim([0, 1])
    plt.ylabel('True Positive Rate')
    plt.xlabel('False Positive Rate')
    plt.show()

def print_Accuracy( TITLE, LIST ) :
    print( TITLE )
    print( "======" )
    for theResults in LIST :
        NAME = theResults[0]
        ACC = theResults[1]
        print( NAME, " = ", ACC )
    print( "------\n\n" )

def getAmtAccuracyScores( NAME, MODEL, X, Y ) :
    pred = MODEL.predict( X )
    MEAN = Y.mean()
    RMSE = math.sqrt( metrics.mean_squared_error( Y, pred))
    return [NAME, RMSE, MEAN]



In [ ]:
# Single Tree
def getTreeVars( TREE, varNames ) :
    tree_ = TREE.tree_
    varName = [ varNames[i] if i != _tree.TREE_UNDEFINED else "undefined!" for i in tree_.feature ]

    nameSet = set()
    for i in tree_.feature :
        if i != _tree.TREE_UNDEFINED :
            nameSet.add( i )
    nameList = list( nameSet )
    parameter_list = list()
    for i in nameList :
        parameter_list.append( varNames[i] )
    return parameter_list

WHO = "TREE"

AMT = tree.DecisionTreeRegressor( max_depth=4 )
AMT = AMT.fit( W_train, Z_train[TARGET_A] )

TRAIN_AMT = getAmtAccuracyScores( WHO + "_Train", AMT, W_train, Z_train[TARGET_A] )
TEST_AMT = getAmtAccuracyScores( WHO, AMT, W_test, Z_test[TARGET_A] )

print_Accuracy( WHO + " RMSE ACCURACY", [ TRAIN_AMT, TEST_AMT ] )

feature_cols = list( X.columns.values )
vars_tree_amt = getTreeVars( AMT, feature_cols ) 
# tree.export_graphviz(AMT,out_file='tree_a.txt',filled=True, rounded=True, feature_names = feature_cols, impurity=False, precision=0  )

TREE_AMT = TEST_AMT.copy()

TREE RMSE ACCURACY
TREE_Train  =  0.18544199085308188
TREE  =  0.2209455800245225
------




In [11]:

# RANDOM FOREST
def getEnsembleTreeVars( ENSTREE, varNames ) :
    importance = ENSTREE.feature_importances_
    index = np.argsort(importance)
    theList = []
    for i in index :
        imp_val = importance[i]
        if imp_val > np.average( ENSTREE.feature_importances_ ) :
            v = int( imp_val / np.max( ENSTREE.feature_importances_ ) * 100 )
            theList.append( ( varNames[i], v ) )
    theList = sorted(theList,key=itemgetter(1),reverse=True)
    return theList

WHO = "RF"

AMT = RandomForestRegressor(n_estimators = 100, random_state=5)
AMT = AMT.fit( W_train, Z_train[TARGET_A] )

TRAIN_AMT = getAmtAccuracyScores( WHO + "_Train", AMT, W_train, Z_train[TARGET_A] )
TEST_AMT = getAmtAccuracyScores( WHO, AMT, W_test, Z_test[TARGET_A] )

print_Accuracy( WHO + " RMSE ACCURACY", [ TRAIN_AMT, TEST_AMT ] )

feature_cols = list( X.columns.values )
vars_RF_amt = getEnsembleTreeVars( AMT, feature_cols )

# for i in vars_RF_amt :
#     print( i )

RF_AMT = TEST_AMT.copy()

RF RMSE ACCURACY
RF_Train  =  0.05424164952969402
RF  =  0.1423909417385709
------




In [12]:
# GRADIENT BOOSTING

WHO = "GB"


AMT = GradientBoostingRegressor(random_state=5)
AMT = AMT.fit( W_train, Z_train[TARGET_A] )

TRAIN_AMT = getAmtAccuracyScores( WHO + "_Train", AMT, W_train, Z_train[TARGET_A] )
TEST_AMT = getAmtAccuracyScores( WHO, AMT, W_test, Z_test[TARGET_A] )

print_Accuracy( WHO + " RMSE ACCURACY", [ TRAIN_AMT, TEST_AMT ] )

feature_cols = list( X.columns.values )
vars_GB_amt = getEnsembleTreeVars( AMT, feature_cols )

##for i in vars_GB_amt :
##    print( i )

GB_AMT = TEST_AMT.copy()

GB RMSE ACCURACY
GB_Train  =  0.07948379391470245
GB  =  0.11995632756534773
------




In [14]:

# REGRESSION FUNCTIONS

def getCoefLogit( MODEL, TRAIN_DATA ) :
    varNames = list( TRAIN_DATA.columns.values )
    coef_dict = {}
    coef_dict["INTERCEPT"] = MODEL.intercept_[0]
    for coef, feat in zip(MODEL.coef_[0],varNames):
        coef_dict[feat] = coef
    print("\nLOAN DEFAULT")
    print("---------")
    print("Total Variables: ", len( coef_dict ) )
    for i in coef_dict :
        print( i, " = ", coef_dict[i]  )


def getCoefLinear( MODEL, TRAIN_DATA ) :
    varNames = list( TRAIN_DATA.columns.values )
    coef_dict = {}
    coef_dict["INTERCEPT"] = MODEL.intercept_
    for coef, feat in zip(MODEL.coef_,varNames):
        coef_dict[feat] = coef
    print("\nLOSS AMOUNT")
    print("---------")
    print("Total Variables: ", len( coef_dict ) )
    for i in coef_dict :
        print( i, " = ", coef_dict[i]  )
        

# REGRESSION ALL VARIABLES

WHO = "REG_ALL"


AMT = LinearRegression()
AMT = AMT.fit( W_train, Z_train[TARGET_A] )

TRAIN_AMT = getAmtAccuracyScores( WHO + "_Train", AMT, W_train, Z_train[TARGET_A] )
TEST_AMT = getAmtAccuracyScores( WHO, AMT, W_test, Z_test[TARGET_A] )

print_Accuracy( WHO + " RMSE ACCURACY", [ TRAIN_AMT, TEST_AMT ] )

varNames = list( X_train.columns.values )

# REG_ALL_CLM_COEF = getCoefLogit( CLM, X_train )
# REG_ALL_AMT_COEF = getCoefLinear( AMT, X_train )


REG_ALL_AMT = TEST_AMT.copy()

REG_ALL RMSE ACCURACY
REG_ALL_Train  =  0.08806251755201193
REG_ALL  =  0.13161394932110085
------




In [15]:
"""
REGRESSION DECISION TREE
"""

WHO = "REG_TREE"

AMT = LinearRegression()
AMT = AMT.fit( W_train[vars_tree_amt], Z_train[TARGET_A] )

TRAIN_AMT = getAmtAccuracyScores( WHO + "_Train", AMT, W_train[vars_tree_amt], Z_train[TARGET_A] )
TEST_AMT = getAmtAccuracyScores( WHO, AMT, W_test[vars_tree_amt], Z_test[TARGET_A] )

print_Accuracy( WHO + " RMSE ACCURACY", [ TRAIN_AMT, TEST_AMT ] )

varNames = list( X_train.columns.values )

#REG_TREE_CLM_COEF = getCoefLogit( CLM, X_train[vars_tree_flag] )
#REG_TREE_AMT_COEF = getCoefLinear( AMT, X_train[vars_tree_amt] )

REG_TREE_AMT = TEST_AMT.copy()


REG_TREE RMSE ACCURACY
REG_TREE_Train  =  0.15771630162804448
REG_TREE  =  0.15540482943540554
------




In [17]:
"""
REGRESSION RANDOM FOREST

"""


print("\n\n")
RF_amt = []
for i in vars_RF_amt :
    print(i)
    theVar = i[0]
    RF_amt.append( theVar )

    
WHO = "REG_RF"

AMT = LinearRegression()
AMT = AMT.fit( W_train[RF_amt], Z_train[TARGET_A] )

TRAIN_AMT = getAmtAccuracyScores( WHO + "_Train", AMT, W_train[RF_amt], Z_train[TARGET_A] )
TEST_AMT = getAmtAccuracyScores( WHO, AMT, W_test[RF_amt], Z_test[TARGET_A] )

print_Accuracy( WHO + " RMSE ACCURACY", [ TRAIN_AMT, TEST_AMT ] )

# REG_RF_CLM_COEF = getCoefLogit( CLM, X_train[RF_flag] )
# REG_RF_AMT_COEF = getCoefLinear( AMT, X_train[RF_amt] )

REG_RF_AMT = TEST_AMT.copy()




('TRUNC_OverallQual', 100)
('TRUNC_GrLivArea', 12)
('TRUNC_GarageCars', 8)
('TRUNC_TotalBsmtSF', 6)
('TRUNC_BsmtFinSF1', 4)
('TRUNC_GarageArea', 4)
('TRUNC_1stFlrSF', 3)
('YearBuilt', 2)
('TRUNC_LotArea', 2)
('TRUNC_BsmtUnfSF', 1)
('TRUNC_2ndFlrSF', 1)
('YearRemodAdd', 1)
('TRUNC_OverallCond', 1)
('O_z_ExterCond_Fa', 0)
('TRUNC_MSSubClass', 0)
('z_IMP_GarageCond_TA', 0)
('z_IMP_BsmtQual_Gd', 0)
('TRUNC_BedroomAbvGr', 0)
('z_IMP_GarageType_Attchd', 0)
('z_IMP_GarageType_Detchd', 0)
('TRUNC_IMP_MasVnrArea', 0)
('TRUNC_TotRmsAbvGrd', 0)
('MoSold', 0)
('O_z_MSZoning_C (all)', 0)
('TRUNC_WoodDeckSF', 0)
('IMP_GarageYrBlt', 0)
('TRUNC_OpenPorchSF', 0)
('TRUNC_Fireplaces', 0)
('z_IMP_FireplaceQu_MISSING', 0)
('TRUNC_FullBath', 0)
('TRUNC_IMP_LotFrontage', 0)
('TRUNC_z_MSZoning_RM', 0)
REG_RF RMSE ACCURACY
REG_RF_Train  =  0.13190579694862853
REG_RF  =  0.12707130199210065
------




In [19]:
"""
REGRESSION GRADIENT BOOSTING
"""

WHO = "REG_GB"

print("\n\n")
GB_amt = []
for i in vars_GB_amt :
    print(i)
    theVar = i[0]
    GB_amt.append( theVar )


AMT = LinearRegression()
AMT = AMT.fit( W_train[GB_amt], Z_train[TARGET_A] )

TRAIN_AMT = getAmtAccuracyScores( WHO + "_Train", AMT, W_train[GB_amt], Z_train[TARGET_A] )
TEST_AMT = getAmtAccuracyScores( WHO, AMT, W_test[GB_amt], Z_test[TARGET_A] )

# print_Accuracy( WHO + " RMSE ACCURACY", [ TRAIN_AMT, TEST_AMT ] )

REG_GB_AMT_COEF = getCoefLinear( AMT, X_train[GB_amt] )

REG_GB_AMT = TEST_AMT.copy()




('TRUNC_OverallQual', 100)
('TRUNC_GrLivArea', 28)
('TRUNC_TotalBsmtSF', 10)
('TRUNC_GarageCars', 10)
('YearBuilt', 5)
('TRUNC_BsmtFinSF1', 5)
('TRUNC_LotArea', 3)
('TRUNC_Fireplaces', 3)
('TRUNC_OverallCond', 3)
('YearRemodAdd', 3)
('TRUNC_1stFlrSF', 2)
('TRUNC_FullBath', 1)
('TRUNC_z_MSZoning_RM', 1)
('z_KitchenQual_TA', 1)
('TRUNC_GarageArea', 1)
('TRUNC_z_IMP_BsmtExposure_Gd', 0)
('z_IMP_GarageType_Detchd', 0)
('z_MSZoning_RL', 0)
('TRUNC_BsmtFullBath', 0)
('TRUNC_z_IMP_BsmtQual_Ex', 0)
('TRUNC_ScreenPorch', 0)
('TRUNC_WoodDeckSF', 0)
('TRUNC_z_LandContour_Bnk', 0)
('z_IMP_GarageType_Attchd', 0)
('TRUNC_z_KitchenQual_Ex', 0)
('z_KitchenQual_Gd', 0)
('TRUNC_2ndFlrSF', 0)
('TRUNC_z_Neighborhood_Crawfor', 0)
('O_z_MSZoning_C (all)', 0)
('IMP_GarageYrBlt', 0)
('TRUNC_z_Neighborhood_Edwards', 0)
('z_IMP_GarageQual_TA', 0)
('TRUNC_HalfBath', 0)
('TRUNC_z_CentralAir_N', 0)

LOSS AMOUNT
---------
Total Variables:  35
INTERCEPT  =  4.809306723113525
TRUNC_OverallQual  =  0.05941516225949